# Exp 25b Colab Runner

Self-contained Colab notebook for running the Exp 25b disformal validation grid when the local GX10 is unavailable.

Protocol discipline:
- The notebook pins a repo commit before running.
- The first execution step is V0-only.
- The full measurement runs only when `RUN_FULL = True`.
- The result JSON, stdout log, and provenance manifest are copied to Google Drive.


In [ ]:
# Configuration
REPO_URL = "https://github.com/Editorenbici/rendering-universe.git"
COMMIT = "59acb3c"  # Exp 25b runner v2 audited target
WORKDIR = "/content/rendering-universe"
DRIVE_OUT = "/content/drive/MyDrive/render_universe_exp25b"
RUN_FULL = True  # Keep False for smoke/V0-only checks. Set True only after author approval.


In [ ]:
# Mount Drive for persistent outputs.
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Clone the public repo and pin the audited commit.
import os, shutil, subprocess, textwrap, json, hashlib, datetime, pathlib

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)
subprocess.run(["git", "checkout", COMMIT], cwd=WORKDIR, check=True)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=WORKDIR, text=True))


In [ ]:
# Provenance: script hash, Python/numpy versions, and prereg path.
import sys, numpy as np
script = pathlib.Path(WORKDIR) / "code" / "analysis" / "25b_disformal_validation.py"
prereg = pathlib.Path(WORKDIR) / "notes" / "foundations" / "exp25b_prereg_2026-07-06.md"
sha = hashlib.sha256(script.read_bytes()).hexdigest()
print("script:", script)
print("script_sha256:", sha)
print("python:", sys.version)
print("numpy:", np.__version__)
print("prereg_exists:", prereg.exists())


In [ ]:
# V0-only audit gate. This must pass before the full grid.
v0 = subprocess.run(
    [sys.executable, "code/analysis/25b_disformal_validation.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(v0.stdout)
print(v0.stderr)
assert v0.returncode == 0, "V0 failed; do not run --run"


In [ ]:
# Full Exp 25b grid. This may take a long time on Colab CPU.
assert RUN_FULL, "RUN_FULL is False; stopping after V0 by design."
log_path = pathlib.Path(WORKDIR) / "outputs" / "exp25b_colab_stdout.log"
log_path.parent.mkdir(exist_ok=True)
cmd = [sys.executable, "-u", "code/analysis/25b_disformal_validation.py", "--run"]
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    rc = proc.wait()
assert rc == 0, f"Exp 25b --run failed with exit code {rc}"


In [ ]:
# Copy JSON, stdout log, and manifest to Drive.
out_dir = pathlib.Path(DRIVE_OUT)
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
json_src = pathlib.Path(WORKDIR) / "outputs" / "exp25b_results.json"
log_src = pathlib.Path(WORKDIR) / "outputs" / "exp25b_colab_stdout.log"
json_dst = out_dir / f"exp25b_results_{COMMIT}_{stamp}.json"
log_dst = out_dir / f"exp25b_stdout_{COMMIT}_{stamp}.log"
manifest_dst = out_dir / f"exp25b_manifest_{COMMIT}_{stamp}.json"
shutil.copy2(json_src, json_dst)
shutil.copy2(log_src, log_dst)
manifest = {
    "repo_url": REPO_URL,
    "commit": COMMIT,
    "script_sha256": sha,
    "python": sys.version,
    "numpy": np.__version__,
    "utc_finished": stamp,
    "json": str(json_dst),
    "stdout_log": str(log_dst),
}
manifest_dst.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
